# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.gitbook.io/) library, referencing all dataset entities by their `@id` as per the Croissant specification.

### Dataset Source

The dataset is defined by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install the mlcroissant library if it's not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from FAIR² using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Authors: {[author['@id'] for author in metadata.author]}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Explore the dataset's available record sets, fields, and their `@id` values. This ensures each entity is uniquely referenced.

In [ ]:
# List all record sets using their @id
# Croissant datasets may define record sets with their @id; let's enumerate these via the metadata.
def get_record_set_ids(dataset):
    rs_ids = []
    # Many Croissant datasets store record sets in their metadata.record_set
    rec_sets = getattr(dataset.metadata, 'record_set', [])
    for rec_set in rec_sets:
        # rec_set can be dicts or string @id
        if isinstance(rec_set, dict) and '@id' in rec_set:
            rs_ids.append(rec_set['@id'])
        elif isinstance(rec_set, str):
            rs_ids.append(rec_set)
    return rs_ids

record_set_ids = get_record_set_ids(dataset)

if not record_set_ids:
    print("No explicit record sets found in metadata; attempting to infer from files/fields.")
    # Fallback: mlcroissant will infer from available records() API
    ds_record_sets = dataset._record_sets
    record_set_ids = [rs['@id'] for rs in ds_record_sets]
    print(f"Record set @ids found: {record_set_ids}")
else:
    print(f"Record set @ids found: {record_set_ids}")

# For each record set, print the available fields and their @id
for rsid in record_set_ids:
    print(f"\nFields for record set @id: {rsid}")
    # Try to get fields via the API
    records = list(dataset.records(record_set=rsid, limit=1))
    if records:
        print(f"Sample record keys: {list(records[0].keys())}")
    else:
        print('No records found (possibly file not present or not public).')

## 3. Data Extraction

Load one or more record sets by their `@id` into pandas DataFrames for further analysis.

Below, we'll load each detected record set and show their columns and sample records corresponding to their field `@id` values.

In [ ]:
# Extract all detected record sets into DataFrames, indexed by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields (@id as DataFrame columns): {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records loaded for this record set.")

# For demonstration: Pick one record set @id as the main for EDA. If only one found, use that.
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nUsing main_record_set_id for EDA: {main_record_set_id}")
    print(f"Available fields: {dataframes[main_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)

We'll:
 - reference all fields and columns by their `@id` (as per the DataFrame columns),
 - demonstrate filtering on a numeric column,
 - normalize it,
 - and optionally group results by a categorical field if present.

---
Please **replace** the `<numeric_field_id>` and `<group_field_id>` below with the appropriate `@id` corresponding to the numeric and grouping variables for your record set. For demonstration, we'll attempt to automatically select a likely candidate.


In [ ]:
# EDA: Select numeric and group fields by their @id
if not main_record_set_id:
    print("No main record set loaded.")
else:
    df = dataframes[main_record_set_id]
    # Try to find the first float or integer-like column
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and not numeric_field_id:
            numeric_field_id = col
        elif not group_field_id:
            # Find a non-numeric, hopefully categorical field
            if df[col].dtype == object:
                group_field_id = col
    if not numeric_field_id:
        print("No numeric field found for EDA. Please update with an explicit field @id.")
    else:
        print(f"Using numeric field (@id): {numeric_field_id}")
        print(f"Using group field (@id): {group_field_id}")
        # Filtering records above a threshold (e.g., mean + 1 std, or pick an arbitrary threshold)
        threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered rows with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a group field exists, group by it and calculate means
        if (group_field_id is not None) and (group_field_id in filtered_df.columns):
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped filtered data by {group_field_id} and mean of {numeric_field_id}:")
            display(grouped.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relation (if relevant) to the group field, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This exploration notebook demonstrated how to:
- load FAIR² metadata and records via Croissant using `mlcroissant`,
- enumerate record set and field `@id` for robust reference,
- extract data as pandas DataFrames,
- conduct numeric field filtering, normalization, and grouping (by `@id`),
- visualize distributions and group comparisons,
all while referencing the identifiers (`@id`) provided by the Croissant schema.

You can extend this notebook for deeper analyses by exploring additional record sets, assessing field-level metadata, or joining across record sets via shared `@id` keys.